# Iris Flower Classification

Internship task: predict Iris species from sepal and petal measurements using a Scikit-learn classifier.

This notebook uses the CSV dataset supplied with the task.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 1. Load and inspect the data


In [ ]:
from pathlib import Path
df = pd.read_csv(Path('data') / 'Iris.csv')
print('Shape:', df.shape)
display(df.head())
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nSpecies counts:\n', df['Species'].value_counts())


## 2. Explore the measurements
The plots give a quick view of class balance and how the measurements vary between species.


In [ ]:
sns.countplot(data=df, x='Species')
plt.title('Number of flowers in each species')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

sns.pairplot(df.drop(columns='Id'), hue='Species', diag_kind='hist', corner=True)
plt.show()


## 3. Prepare the data and create train/test sets
The ID column is excluded because it is only a row identifier. Stratification keeps the class proportions similar in both sets.


In [ ]:
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
X = df[features]
y = df['Species'].str.replace('Iris-', '', regex=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Training records:', len(X_train))
print('Testing records:', len(X_test))


## 4. Train the KNN classifier
K-Nearest Neighbours compares a new flower with nearby training examples. Here, k=5. Standardising the measurements is important because KNN uses distances. The pipeline fits the scaler on training data only.


In [ ]:
model = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
model.fit(X_train, y_train)
print('Model training completed.')


## 5. Evaluate the model
Accuracy gives the overall proportion of correct predictions. The classification report provides precision, recall and F1-score for each class.


In [ ]:
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print(classification_report(y_test, y_pred))
labels = ['setosa', 'versicolor', 'virginica']
cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()
print(cm)


## 6. Predict a new flower
The four values must be supplied in this order: sepal length, sepal width, petal length and petal width.


In [ ]:
new_flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=features)
print('Predicted species:', model.predict(new_flower)[0])


## Conclusion
The KNN classifier learns from four flower measurements and predicts one of the three Iris species. With this stratified 80/20 split and random_state=42, it achieves 93.33% test accuracy (28 correct predictions out of 30). This result is based on one split, so cross-validation and comparison with other classifiers would be useful follow-up work.
